# Auditoría de la base imputada

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path("..") / "data" / "procesados"
VARS = ["CO", "NO", "NO2", "NOX", "O3", "PM10", "PM2.5", "PRS",
        "RAINF", "RH", "SO2", "SR", "TOUT", "WSR", "WDR"]

In [2]:
ni = pd.read_csv(DATA / "datos_no_imputados.csv", low_memory=False)
im = pd.read_csv(DATA / "datos_imputados.csv")
print("Sin imputar:", ni.shape)
print("Imputada:   ", im.shape)

Sin imputar: (830904, 33)
Imputada:    (831095, 17)



El Excel de 2024 trae los nombres con unidades (`"CO (ppm)"`), los demás sin unidades (`"CO"`).
Si un año tiene 0 en la primera columna, el imputador lo vio vacío.

In [3]:
con_unidades = [c for c in ni.columns if " (" in c]
pd.DataFrame({
    "datos en columnas sin unidades": ni.groupby("Archivo_Origen")[VARS].count().sum(axis=1),
    "datos en columnas con unidades": ni.groupby("Archivo_Origen")[con_unidades].count().sum(axis=1),
    "filas": ni.groupby("Archivo_Origen").size(),
})

,datos en columnas sin unidades,datos en columnas con unidades,filas
Archivo_Origen,,,
BD 2020.xlsx,1158599,0,114102
BD 2021.xlsx,1641744,0,122629
BD 2022.xlsx,1766600,0,123383
BD 2023.xlsx,1867099,0,131370
BD 2024.xlsx,0,1836531,131741
BD 2025.xlsx,1758598,0,131378
BD 2026.xlsx,1031191,0,76301


3. Juntar las columnas y alinear las dos bases
Pasamos los valores de `"CO (ppm)"` a `"CO"`, etc. Así recuperamos los datos reales de 2024.
Luego unimos por estación y fecha: cada variable queda como `_real` y `_imp`.

In [4]:
for c in con_unidades:
    base = c.split(" (")[0]
    ni[base] = ni[base].fillna(ni[c])

for df in (ni, im):
    df["Fecha y hora"] = pd.to_datetime(df["Fecha y hora"])

m = ni[["Estacion", "Fecha y hora"] + VARS].merge(
        im, on=["Estacion", "Fecha y hora"], suffixes=("_real", "_imp"))
m["año"] = m["Fecha y hora"].dt.year
print(m.shape)

(830904, 33)



Donde había dato real, la base imputada debería traer exactamente el mismo valor (0 % cambiados).

In [5]:
def pct_cambiados(g, v):
    k = g[v + "_real"].notna()
    return 100 * ((g.loc[k, v + "_real"] - g.loc[k, v + "_imp"]).abs() > 1e-3).mean()

m.groupby("año").apply(lambda g: pd.Series({v: pct_cambiados(g, v) for v in ["O3", "PM10", "SO2"]})).round(1)

,O3,PM10,SO2
año,,,
2020,0.0,0.0,0.0
2021,0.0,0.0,0.0
2022,0.0,0.0,0.0
2023,0.0,0.0,0.0
2024,100.0,100.0,100.0
2025,0.0,0.0,0.0
2026,0.0,0.0,0.0


2024 como prueba: real contra imputado

Es una validación natural: si la imputación fuera buena, la correlación sería alta y los extremos se parecerían.

In [6]:
a = m[m["año"] == 2024]
filas = {}
for v in VARS:
    k = a[v + "_real"].notna()
    r, i = a.loc[k, v + "_real"], a.loc[k, v + "_imp"]
    filas[v] = {"correlación": r.corr(i), "media real": r.mean(), "media imputada": i.mean(),
                "sesgo": (i - r).mean(), "P99 real": r.quantile(0.99), "P99 imputado": i.quantile(0.99)}
pd.DataFrame(filas).T.round(2)

,correlación,media real,media imputada,sesgo,P99 real,P99 imputado
CO,0.05,1.50,1.55,0.05,4.58,1.85
NO,0.08,12.04,12.47,0.43,102.20,27.93
NO2,0.14,14.86,5.86,-9.00,50.30,11.54
NOX,0.13,26.67,20.58,-6.10,140.10,42.45
O3,0.25,28.02,15.96,-12.06,92.00,25.25
PM10,0.11,62.62,42.01,-20.61,213.07,55.90
PM2.5,0.11,20.60,16.21,-4.39,74.00,21.93
PRS,-0.00,714.75,723.13,8.38,735.90,727.45
RAINF,0.00,0.02,-0.16,-0.18,0.13,0.46
RH,0.04,57.59,68.93,11.34,92.00,73.57


 ¿Cuántos excesos ve el modelo?
La red solo aprende de los excesos sobre el umbral (O3: 63 ppb, SO2: 20.2 ppb).

In [7]:
UMBRALES = {"O3": 63.0, "SO2": 20.2}
pd.concat({v: m.groupby("año").apply(lambda g: pd.Series({
              "real": (g[v + "_real"] > u).sum(),
              "base imputada": (g[v + "_imp"] > u).sum()}))
           for v, u in UMBRALES.items()}, axis=1)

O3                 SO2              
      real base imputada  real base imputada
año                                         
2020  2177          2177  1204          1204
2021  4541          4541  1296          1296
2022  4757          4757  1239          1239
2023  5182          5183   700           700
2024  7190             0   734             0
2025  8596          8596  1289          1289
2026  3987          3988   420           420

7. Valores físicamente imposibles en la base imputada

In [8]:
I = {v: m[v + "_imp"] for v in VARS}
pd.Series({
    "Lluvia negativa": (I["RAINF"] < 0).sum(),
    "Radiación negativa": (I["SR"] < 0).sum(),
    "Contaminantes negativos": sum((I[v] < 0).sum() for v in ["CO", "NO", "NO2", "NOX", "O3", "PM10", "PM2.5", "SO2"]),
    "RH > 100 %": (I["RH"] > 100).sum(),
    "PM2.5 > PM10": (I["PM2.5"] > I["PM10"]).sum(),
    "Temperatura fuera de [-10, 50]": ((I["TOUT"] < -10) | (I["TOUT"] > 50)).sum(),
    "PM10 >= 999": (I["PM10"] >= 999).sum(),
}, name="casos")

Lluvia negativa                   128283
Radiación negativa                148051
Contaminantes negativos            10193
RH > 100 %                           267
PM2.5 > PM10                       11686
Temperatura fuera de [-10, 50]       427
PM10 >= 999                            4
Name: casos, dtype: int64

¿La imputación aplana los extremos? (fuera de 2024)
Comparamos los valores observados contra los que el modelo rellenó en los huecos.

In [9]:
fuera = m["año"] != 2024
filas = {}
for v in ["O3", "PM10", "PM2.5", "NO2", "SO2", "CO"]:
    obs = m.loc[fuera, v + "_real"].dropna()
    imp = m.loc[fuera & m[v + "_real"].isna(), v + "_imp"]
    filas[v] = {"n imputados": len(imp), "media obs": obs.mean(), "media imp": imp.mean(),
                "P99 obs": obs.quantile(0.99), "P99 imp": imp.quantile(0.99), "máx imp": imp.max()}
pd.DataFrame(filas).T.round(2)

,n imputados,media obs,media imp,P99 obs,P99 imp,máx imp
O3,106811.0,27.27,19.21,88.00,34.57,69.40
PM10,31735.0,54.99,46.17,195.00,114.23,197.18
PM2.5,158524.0,19.99,16.14,69.00,43.19,91.78
NO2,131526.0,14.22,8.76,53.20,25.66,84.22
SO2,106285.0,5.01,5.29,20.60,8.70,12.16
CO,100319.0,1.29,1.14,3.77,2.42,3.37
